#  Multimodal Human Emotion, Engagement & Stress Detection System
### For Online Learning Analytics

**Datasets used:**
- **facedata** – FER-style facial emotion images (train/test split, 7 classes)
- **RAVDESS** – Audio emotion WAV files (Actor folders)
- **ISEAR.csv** – Text emotion dataset (CSV)

**Pipeline:**
1. Install dependencies
2. Mount Google Drive & load datasets
3. Train Face Emotion CNN
4. Train Voice Emotion CNN on MFCCs
5. Train Text Emotion DistilBERT
6. Build Fusion Engine
7. Launch Interactive Dashboard UI

## Step 1: Install Dependencies

In [ ]:
# Install all required packages
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers datasets librosa soundfile opencv-python-headless scikit-learn matplotlib seaborn Pillow
!pip install -q ipywidgets ipython gradio
print('All packages installed!')

In [ ]:
import os

BASE_PATH    = '/content/drive/MyDrive' 
FACE_TRAIN   = os.path.join(BASE_PATH, 'data/facedata/train')
FACE_TEST    = os.path.join(BASE_PATH, 'data/facedata/test')
RAVDESS_PATH = os.path.join(BASE_PATH, 'data/RAVDESS')
ISEAR_PATH   = os.path.join(BASE_PATH, 'data/ISEAR/ISEAR.csv')
MODEL_SAVE   = os.path.join(BASE_PATH, 'emotion_models')
os.makedirs(MODEL_SAVE, exist_ok=True)

for label, path in [('Face Train', FACE_TRAIN), ('Face Test', FACE_TEST),
                    ('RAVDESS', RAVDESS_PATH), ('ISEAR CSV', ISEAR_PATH)]:
    exists = os.path.exists(path)
    print(f"{'✅' if exists else '❌'} {label}: {path}")

## Step 3: Global Imports & Config

In [ ]:
import os, glob, re, json, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

import librosa
import librosa.display
import soundfile as sf

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import get_linear_schedule_with_warmup

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')


FACE_EMOTIONS  = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

RAVDESS_MAP = {
    '01': 'neutral', '02': 'calm',    '03': 'happy',
    '04': 'sad',     '05': 'angry',   '06': 'fearful',
    '07': 'disgust', '08': 'surprised'
}

UNIFIED_EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

RAVDESS_TO_UNIFIED = {
    'neutral':'neutral', 'calm':'neutral', 'happy':'happy',
    'sad':'sad', 'angry':'angry', 'fearful':'fear',
    'disgust':'disgust', 'surprised':'surprise'
}

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
print('Imports complete!')

## Step 4: Face Emotion CNN — Data Preparation

In [ ]:
class FaceEmotionDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples  = []
        self.labels   = []
        self.transform = transform
        self.class_to_idx = {c: i for i, c in enumerate(FACE_EMOTIONS)}
        for cls in FACE_EMOTIONS:
            cls_dir = os.path.join(root_dir, cls)
            if not os.path.isdir(cls_dir):
                print(f'⚠️  Missing class folder: {cls_dir}')
                continue
            for ext in ('*.jpg','*.jpeg','*.png'):
                for fpath in glob.glob(os.path.join(cls_dir, ext)):
                    self.samples.append(fpath)
                    self.labels.append(self.class_to_idx[cls])
        print(f'   Loaded {len(self.samples)} images from {root_dir}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.samples[idx]).convert('RGB')
        except Exception:
            img = Image.new('RGB', (48, 48))
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

face_train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])
face_test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

print('Loading face datasets...')
face_train_ds = FaceEmotionDataset(FACE_TRAIN, face_train_transform)
face_test_ds  = FaceEmotionDataset(FACE_TEST,  face_test_transform)

face_train_loader = DataLoader(face_train_ds, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
face_test_loader  = DataLoader(face_test_ds,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
print(f'✅ Train: {len(face_train_ds)} | Test: {len(face_test_ds)}')

In [ ]:
fig, axes = plt.subplots(1, 7, figsize=(18, 3))
fig.suptitle('Face Dataset — Sample per Class', fontsize=14, fontweight='bold')
for i, cls in enumerate(FACE_EMOTIONS):
    cls_dir = os.path.join(FACE_TRAIN, cls)
    files = glob.glob(os.path.join(cls_dir, '*.jpg')) + glob.glob(os.path.join(cls_dir, '*.png'))
    if files:
        img = Image.open(random.choice(files)).convert('RGB').resize((64,64))
        axes[i].imshow(img)
    axes[i].set_title(cls, fontsize=10)
    axes[i].axis('off')
plt.tight_layout()
plt.savefig('/content/face_samples.png', dpi=100, bbox_inches='tight')
plt.show()
print('Face sample grid saved.')

## Step 5: Face Emotion CNN Model — Build & Train

In [ ]:
class FaceEmotionCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

face_model = FaceEmotionCNN(num_classes=7).to(DEVICE)
total_params = sum(p.numel() for p in face_model.parameters())
print(f'✅ Face CNN ready | Params: {total_params:,}')

In [ ]:
def train_model(model, train_loader, test_loader, num_epochs=50,
                lr=1e-3, model_name='model', save_path=MODEL_SAVE):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
    best_val_acc = 0.0

    for epoch in range(num_epochs):
        # ── Train ──
        model.train()
        t_loss, t_correct, t_total = 0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            t_loss    += loss.item() * imgs.size(0)
            t_correct += (out.argmax(1) == labels).sum().item()
            t_total   += imgs.size(0)
        scheduler.step()

        model.eval()
        v_loss, v_correct, v_total = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                out  = model(imgs)
                loss = criterion(out, labels)
                v_loss    += loss.item() * imgs.size(0)
                v_correct += (out.argmax(1) == labels).sum().item()
                v_total   += imgs.size(0)

        t_acc = t_correct / t_total
        v_acc = v_correct / v_total
        history['train_loss'].append(t_loss / t_total)
        history['train_acc'].append(t_acc)
        history['val_loss'].append(v_loss / v_total)
        history['val_acc'].append(v_acc)

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            torch.save(model.state_dict(), os.path.join(save_path, f'{model_name}_best.pth'))

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'Epoch {epoch+1:3d}/{num_epochs} | '
                  f'Train Loss: {t_loss/t_total:.4f} Acc: {t_acc:.4f} | '
                  f'Val Loss: {v_loss/v_total:.4f} Acc: {v_acc:.4f}')

    print(f'\n Best Val Accuracy: {best_val_acc:.4f}')
    return history

print('Starting Face CNN training...')
face_history = train_model(face_model, face_train_loader, face_test_loader,
                           num_epochs=30, lr=1e-3, model_name='face_cnn')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Face CNN Training', fontsize=13, fontweight='bold')
ax1.plot(face_history['train_loss'], label='Train Loss', color='#7c3aed')
ax1.plot(face_history['val_loss'],   label='Val Loss',   color='#f59e0b')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.set_title('Loss')
ax2.plot(face_history['train_acc'], label='Train Acc', color='#7c3aed')
ax2.plot(face_history['val_acc'],   label='Val Acc',   color='#f59e0b')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.set_title('Accuracy')
plt.tight_layout()
plt.savefig('/content/face_training.png', dpi=100)
plt.show()

In [ ]:
face_model.load_state_dict(torch.load(os.path.join(MODEL_SAVE, 'face_cnn_best.pth'), map_location=DEVICE))
face_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in face_test_loader:
        imgs = imgs.to(DEVICE)
        out  = face_model(imgs)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
print(classification_report(all_labels, all_preds, target_names=FACE_EMOTIONS))
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=FACE_EMOTIONS, yticklabels=FACE_EMOTIONS, ax=ax)
ax.set_title('Face CNN Confusion Matrix', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('/content/face_cm.png', dpi=100)
plt.show()

## Step 6: Voice Emotion CNN — Data Preparation (RAVDESS)

In [ ]:
def extract_mfcc(wav_path, sr=22050, n_mfcc=40, max_pad=174):
    try:
        y, s = librosa.load(wav_path, sr=sr)
        mfcc = librosa.feature.mfcc(y=y, sr=s, n_mfcc=n_mfcc)
        if mfcc.shape[1] < max_pad:
            mfcc = np.pad(mfcc, ((0,0),(0, max_pad - mfcc.shape[1])), mode='constant')
        else:
            mfcc = mfcc[:, :max_pad]
        return mfcc 
    except Exception as e:
        return np.zeros((n_mfcc, max_pad))

def load_ravdess(ravdess_path):
    X, y = [], []
    actor_dirs = sorted(glob.glob(os.path.join(ravdess_path, 'Actor_*')))
    print(f'Found {len(actor_dirs)} actor folders')
    for actor_dir in actor_dirs:
        for wav in glob.glob(os.path.join(actor_dir, '*.wav')):
            fname   = os.path.basename(wav)
            parts   = fname.replace('.wav','').split('-')
            if len(parts) < 3:
                continue
            emo_code = parts[2]         
            emo_raw  = RAVDESS_MAP.get(emo_code)
            if emo_raw is None:
                continue
            emo_unified = RAVDESS_TO_UNIFIED.get(emo_raw)
            if emo_unified not in UNIFIED_EMOTIONS:
                continue
            mfcc = extract_mfcc(wav)
            X.append(mfcc)
            y.append(UNIFIED_EMOTIONS.index(emo_unified))
    print(f' RAVDESS: {len(X)} samples loaded')
    return np.array(X), np.array(y)

X_rav, y_rav = load_ravdess(RAVDESS_PATH)
print(f'Shape: {X_rav.shape}, Labels: {np.unique(y_rav, return_counts=True)}')

In [ ]:
from sklearn.model_selection import train_test_split
X_train_r, X_val_r, y_train_r, y_val_r = train_test_split(
    X_rav, y_rav, test_size=0.2, stratify=y_rav, random_state=SEED)
print(f'Train: {X_train_r.shape} | Val: {X_val_r.shape}')

class AudioDataset(Dataset):
    def __init__(self, X, y):
       
        self.X = torch.FloatTensor(X).unsqueeze(1)
        self.y = torch.LongTensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

audio_train_loader = DataLoader(AudioDataset(X_train_r, y_train_r), batch_size=32, shuffle=True)
audio_val_loader   = DataLoader(AudioDataset(X_val_r,   y_val_r),   batch_size=32, shuffle=False)
print('Audio DataLoaders ready!')

## Step 7: Voice Emotion CNN Model — Build & Train

In [ ]:
class VoiceEmotionCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, (3,3), padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d((2,2)),
            nn.Conv2d(32, 64, (3,3), padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d((2,2)),
            nn.Conv2d(64, 128, (3,3), padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(128*4*4, 256), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    def forward(self, x): return self.net(x)

voice_model = VoiceEmotionCNN(num_classes=7).to(DEVICE)
print(f'✅ Voice CNN ready | Params: {sum(p.numel() for p in voice_model.parameters()):,}')

voice_history = train_model(voice_model, audio_train_loader, audio_val_loader,
                            num_epochs=40, lr=1e-3, model_name='voice_cnn')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Voice CNN Training', fontsize=13, fontweight='bold')
ax1.plot(voice_history['train_loss'], label='Train', color='#7c3aed')
ax1.plot(voice_history['val_loss'],   label='Val',   color='#f59e0b')
ax1.set_title('Loss'); ax1.legend()
ax2.plot(voice_history['train_acc'], label='Train', color='#7c3aed')
ax2.plot(voice_history['val_acc'],   label='Val',   color='#f59e0b')
ax2.set_title('Accuracy'); ax2.legend()
plt.tight_layout()
plt.savefig('/content/voice_training.png', dpi=100)
plt.show()

##  Step 8: Text Emotion — DistilBERT (ISEAR Dataset)

In [ ]:
df = pd.read_csv(ISEAR_PATH, on_bad_lines='skip')
print('Columns:', df.columns.tolist())
print(df.head())

In [ ]:
text_col  = None
label_col = None

for col in df.columns:
    sample = str(df[col].dropna().iloc[0]).lower() if len(df[col].dropna()) > 0 else ''
    if len(sample) > 30:
        text_col = col
    if sample in ['joy','anger','fear','sadness','disgust','shame','guilt']:
        label_col = col

if text_col is None or label_col is None:
   
    label_col = df.columns[0]
    text_col  = df.columns[-1]

print(f'Text column: {text_col!r}  |  Label column: {label_col!r}')
df = df[[text_col, label_col]].dropna()
df.columns = ['text', 'emotion']
df['text']    = df['text'].astype(str).str.strip()
df['emotion'] = df['emotion'].astype(str).str.strip().str.lower()


ISEAR_MAP = {
    'joy':'happy', 'happiness':'happy', 'happy':'happy',
    'anger':'angry', 'angry':'angry',
    'fear':'fear', 'fearful':'fear',
    'sadness':'sad', 'sad':'sad',
    'disgust':'disgust',
    'shame':'sad', 'guilt':'sad',
    'surprise':'surprise', 'surprised':'surprise',
    'neutral':'neutral'
}
df['emotion'] = df['emotion'].map(ISEAR_MAP)
df = df[df['emotion'].isin(UNIFIED_EMOTIONS)].reset_index(drop=True)
print(f'ISEAR cleaned: {len(df)} samples')
print(df['emotion'].value_counts())

In [ ]:
df['label'] = df['emotion'].map({e: i for i, e in enumerate(UNIFIED_EMOTIONS)})
X_text_train, X_text_val, y_text_train, y_text_val = train_test_split(
    df['text'].tolist(), df['label'].tolist(), test_size=0.2,
    stratify=df['label'], random_state=SEED)
print(f'Train: {len(X_text_train)} | Val: {len(X_text_val)}')


tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

class TextEmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.enc = tokenizer(texts, truncation=True, padding='max_length',
                             max_length=max_len, return_tensors='pt')
        self.labels = torch.LongTensor(labels)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {k: v[i] for k, v in self.enc.items()}, self.labels[i]

text_train_ds = TextEmotionDataset(X_text_train, y_text_train, tokenizer)
text_val_ds   = TextEmotionDataset(X_text_val,   y_text_val,   tokenizer)
text_train_loader = DataLoader(text_train_ds, batch_size=32, shuffle=True)
text_val_loader   = DataLoader(text_val_ds,   batch_size=32, shuffle=False)
print('Text DataLoaders ready!')

In [ ]:
text_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=7).to(DEVICE)
print(f'DistilBERT loaded | Params: {sum(p.numel() for p in text_model.parameters()):,}')

TEXT_EPOCHS = 5
optimizer_text  = torch.optim.AdamW(text_model.parameters(), lr=2e-5)
total_steps     = len(text_train_loader) * TEXT_EPOCHS
scheduler_text  = get_linear_schedule_with_warmup(
    optimizer_text, num_warmup_steps=total_steps//10, num_training_steps=total_steps)

best_text_acc = 0
text_history  = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}

for epoch in range(TEXT_EPOCHS):
    # Train
    text_model.train()
    t_loss, t_correct, t_total = 0, 0, 0
    for batch, labels in text_train_loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = labels.to(DEVICE)
        optimizer_text.zero_grad()
        out  = text_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = out.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(text_model.parameters(), 1.0)
        optimizer_text.step()
        scheduler_text.step()
        t_loss    += loss.item() * labels.size(0)
        t_correct += (out.logits.argmax(1) == labels).sum().item()
        t_total   += labels.size(0)


    text_model.eval()
    v_loss, v_correct, v_total = 0, 0, 0
    with torch.no_grad():
        for batch, labels in text_val_loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = labels.to(DEVICE)
            out  = text_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            v_loss    += out.loss.item() * labels.size(0)
            v_correct += (out.logits.argmax(1) == labels).sum().item()
            v_total   += labels.size(0)

    t_acc, v_acc = t_correct/t_total, v_correct/v_total
    text_history['train_loss'].append(t_loss/t_total)
    text_history['train_acc'].append(t_acc)
    text_history['val_loss'].append(v_loss/v_total)
    text_history['val_acc'].append(v_acc)

    if v_acc > best_text_acc:
        best_text_acc = v_acc
        text_model.save_pretrained(os.path.join(MODEL_SAVE, 'text_distilbert_best'))
        tokenizer.save_pretrained(os.path.join(MODEL_SAVE, 'text_distilbert_best'))

    print(f'Epoch {epoch+1}/{TEXT_EPOCHS} | Train Acc: {t_acc:.4f} | Val Acc: {v_acc:.4f}')

print(f'Best Text Val Acc: {best_text_acc:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('DistilBERT Text Training', fontsize=13, fontweight='bold')
ax1.plot(text_history['train_loss'], label='Train', color='#7c3aed')
ax1.plot(text_history['val_loss'],   label='Val',   color='#f59e0b')
ax1.set_title('Loss'); ax1.legend()
ax2.plot(text_history['train_acc'], label='Train', color='#7c3aed')
ax2.plot(text_history['val_acc'],   label='Val',   color='#f59e0b')
ax2.set_title('Accuracy'); ax2.legend()
plt.tight_layout()
plt.savefig('/content/text_training.png', dpi=100)
plt.show()

## Step 9: Fusion Engine

In [ ]:
class FusionEngine:
    """
    Dynamic weighted fusion of face / voice / text modalities.
    Automatically renormalizes weights when modalities are missing.
    """
    BASE_WEIGHTS = {'face': 0.40, 'voice': 0.35, 'text': 0.25}
    EMOTIONS     = UNIFIED_EMOTIONS

    EMOTION_EMOJI = {
        'angry':'😠', 'disgust':'🤢', 'fear':'😨',
        'happy':'😊', 'neutral':'😐', 'sad':'😢', 'surprise':'😲'
    }

    def fuse(self, face_probs=None, voice_probs=None, text_probs=None):
        """
        Args:
            face_probs  : np.array of shape (7,) or None
            voice_probs : np.array of shape (7,) or None
            text_probs  : np.array of shape (7,) or None
        Returns dict with fused emotion, confidence, modality breakdown.
        """
        active = {}
        if face_probs  is not None: active['face']  = np.array(face_probs)
        if voice_probs is not None: active['voice'] = np.array(voice_probs)
        if text_probs  is not None: active['text']  = np.array(text_probs)

        if not active:
            return {'emotion':'neutral', 'confidence':0.0,
                    'emoji':'😐', 'modalities': {}, 'probs': np.ones(7)/7}

        raw_w   = {k: self.BASE_WEIGHTS[k] for k in active}
        total_w = sum(raw_w.values())
        norm_w  = {k: v/total_w for k, v in raw_w.items()}

        fused = np.zeros(7)
        for mod, probs in active.items():
            fused += norm_w[mod] * probs

        pred_idx  = int(np.argmax(fused))
        emotion   = self.EMOTIONS[pred_idx]
        confidence = float(fused[pred_idx])

        modality_results = {}
        for mod, probs in active.items():
            idx = int(np.argmax(probs))
            modality_results[mod] = {
                'emotion':    self.EMOTIONS[idx],
                'confidence': float(probs[idx]),
                'weight':     norm_w[mod]
            }

        return {
            'emotion':    emotion,
            'confidence': confidence,
            'emoji':      self.EMOTION_EMOJI.get(emotion, '😐'),
            'modalities': modality_results,
            'probs':      fused
        }

    def compute_engagement(self, face_probs=None, text_active=False, voice_active=False):
        """Engagement heuristic 0-100."""
        score = 0
        if face_probs is not None:
            emo_idx = int(np.argmax(face_probs))
            emo = UNIFIED_EMOTIONS[emo_idx]
            face_eng = {'happy':90,'surprise':85,'neutral':60,'sad':30,'angry':20,'fear':20,'disgust':15}
            score += face_eng.get(emo, 50) * 0.5
        else:
            score += 30
        score += (30 if text_active  else 0)
        score += (20 if voice_active else 0)
        return min(100, int(score))

    def compute_stress(self, face_probs=None, voice_probs=None, text_probs=None):
        """Stress heuristic 0-100."""
        stress_emos = {'angry':0.9,'fear':0.85,'disgust':0.7,'sad':0.5}
        score = 0
        counts = 0
        for probs in [face_probs, voice_probs, text_probs]:
            if probs is None: continue
            emo = UNIFIED_EMOTIONS[int(np.argmax(probs))]
            score += stress_emos.get(emo, 0.1) * 100
            counts += 1
        return int(score / counts) if counts else 0


fusion = FusionEngine()

test_face  = np.array([0.1, 0.05, 0.05, 0.6, 0.1, 0.05, 0.05])
test_voice = np.array([0.05, 0.05, 0.1, 0.7, 0.05, 0.0, 0.05])
result = fusion.fuse(face_probs=test_face, voice_probs=test_voice)
print('Fusion test:', result['emoji'], result['emotion'], f"conf={result['confidence']:.3f}")
print('Engagement:', fusion.compute_engagement(face_probs=test_face, text_active=True))
print('Stress:    ', fusion.compute_stress(face_probs=test_face, voice_probs=test_voice))
print('✅ Fusion Engine ready!')

## Step 10: Inference Helpers

In [ ]:
face_model.load_state_dict(torch.load(os.path.join(MODEL_SAVE, 'face_cnn_best.pth'), map_location=DEVICE))
face_model.eval()

voice_model.load_state_dict(torch.load(os.path.join(MODEL_SAVE, 'voice_cnn_best.pth'), map_location=DEVICE))
voice_model.eval()

text_model = DistilBertForSequenceClassification.from_pretrained(
    os.path.join(MODEL_SAVE, 'text_distilbert_best')).to(DEVICE)
tokenizer  = DistilBertTokenizer.from_pretrained(
    os.path.join(MODEL_SAVE, 'text_distilbert_best'))
text_model.eval()

print('All models loaded!')

def predict_face(image_path_or_pil):
    """Returns softmax probabilities (7,) for a face image."""
    try:
        if isinstance(image_path_or_pil, str):
            img = Image.open(image_path_or_pil).convert('RGB')
        else:
            img = image_path_or_pil.convert('RGB')
        img = face_test_transform(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits = face_model(img)
        return F.softmax(logits, dim=1).cpu().numpy()[0]
    except Exception as e:
        print(f'Face predict error: {e}')
        return None

def predict_voice(wav_path):
    """Returns softmax probabilities (7,) for an audio file."""
    try:
        mfcc = extract_mfcc(wav_path)
        inp  = torch.FloatTensor(mfcc).unsqueeze(0).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits = voice_model(inp)
        return F.softmax(logits, dim=1).cpu().numpy()[0]
    except Exception as e:
        print(f'Voice predict error: {e}')
        return None

def predict_text(text):
    """Returns softmax probabilities (7,) for a text string."""
    try:
        enc = tokenizer(text, return_tensors='pt', truncation=True,
                        max_length=128, padding='max_length')
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            logits = text_model(**enc).logits
        return F.softmax(logits, dim=1).cpu().numpy()[0]
    except Exception as e:
        print(f'Text predict error: {e}')
        return None

print('Inference helpers defined!')

## Step 11: Gradio Dashboard — Exact UI Match

In [ ]:
import gradio as gr
import numpy as np
from PIL import Image
import io, base64, soundfile as sf, tempfile, os

# Session state (Gradio is stateless per click; we use gr.State)
session_log = []

EMOJI_MAP = {
    'angry':'😠','disgust':'🤢','fear':'😨',
    'happy':'😊','neutral':'😐','sad':'😢','surprise':'😲','waiting':'😶'
}

CSS = """
:root {
  --purple-dark: #4c1d95;
  --purple-mid:  #6d28d9;
  --purple-light:#8b5cf6;
  --amber:       #f59e0b;
  --green:       #22c55e;
  --red:         #ef4444;
  --bg:          #eef0f7;
  --card:        #ffffff;
}
body { background: var(--bg) !important; font-family: 'Segoe UI', sans-serif; }
.gradio-container { background: var(--bg) !important; }
#header-bar {
  background: linear-gradient(135deg, #6d28d9 0%, #4c1d95 100%);
  color: white; padding: 14px 24px; border-radius: 12px;
  font-size: 1.5em; font-weight: 700; margin-bottom: 16px;
  display: flex; align-items: center; gap: 10px;
}
.panel-card {
  background: var(--card); border-radius: 14px;
  padding: 18px; box-shadow: 0 2px 12px rgba(109,40,217,0.08);
  border: 1px solid #e5e7eb;
}
.section-title {
  font-size:0.75em; font-weight:700; letter-spacing:0.08em;
  color:#6b7280; text-transform:uppercase; margin-bottom:10px;
}
.result-emotion {
  font-size:2em; font-weight:700; color:#6d28d9;
}
.stat-card {
  background:#f5f3ff; border-radius:12px; padding:16px;
  text-align:center; border:1px solid #ddd6fe;
}
.stat-value { font-size:1.6em; font-weight:700; color:#4c1d95; }
.stat-label { font-size:0.8em; color:#6b7280; margin-top:4px; }
.log-table  { font-size:0.82em; }
#analyze-btn {
  background: linear-gradient(135deg, #8b5cf6, #6d28d9) !important;
  color:white !important; font-size:1.1em !important;
  padding: 12px 32px !important; border-radius:30px !important;
  font-weight:600 !important; width:100% !important;
}
"""

def analyze(face_img, voice_file, text_input,
            face_enabled, voice_enabled, text_enabled,
            log_state):
    face_probs = voice_probs = text_probs = None

    if face_enabled and face_img is not None:
        pil = Image.fromarray(face_img.astype('uint8'), 'RGB')
        face_probs = predict_face(pil)

    if voice_enabled and voice_file is not None:
        voice_probs = predict_voice(voice_file)

    if text_enabled and text_input and text_input.strip():
        text_probs = predict_text(text_input.strip())

    result  = fusion.fuse(face_probs, voice_probs, text_probs)
    eng     = fusion.compute_engagement(face_probs, text_active=text_probs is not None,
                                         voice_active=voice_probs is not None)
    stress  = fusion.compute_stress(face_probs, voice_probs, text_probs)

    emo     = result['emotion']
    emoji   = result['emoji']
    conf    = result['confidence']

    mods    = result['modalities']
    face_r  = mods.get('face',  {'emotion':'N/A','confidence':0})
    voice_r = mods.get('voice', {'emotion':'N/A','confidence':0})
    text_r  = mods.get('text',  {'emotion':'N/A','confidence':0})

    result_text = f"{emoji}  **{emo.upper()}**\n\nConfidence: {conf*100:.1f}%\nEngagement: {eng}%\nStress: {stress}%"

    breakdown = (
        f"**FACE**\n{face_r['emotion']}  ({face_r['confidence']*100:.1f}%)\n\n"
        f"**SPEECH**\n{voice_r['emotion']}  ({voice_r['confidence']*100:.1f}%)\n\n"
        f"**TEXT**\n{text_r['emotion']}  ({text_r['confidence']*100:.1f}%)"
    )

    import datetime
    log_state = log_state or []
    log_state.append({
        'time':  datetime.datetime.now().strftime('%H:%M:%S'),
        'face':  face_r['emotion'],
        'voice': voice_r['emotion'],
        'text':  text_r['emotion'],
        'fused': emo,
        'eng':   eng,
        'stress':stress
    })

 
    analyzed   = len(log_state)
    avg_eng    = f"{np.mean([r['eng']    for r in log_state]):.1f}%"
    avg_stress = f"{np.mean([r['stress'] for r in log_state]):.1f}%"
    top_emo    = max(set([r['fused'] for r in log_state]), key=[r['fused'] for r in log_state].count)

    log_md = "| Time | Face | Speech | Text | Fused | Eng | Stress |\n"
    log_md += "|------|------|--------|------|-------|-----|--------|\n"
    for r in log_state[-10:]:
        log_md += f"| {r['time']} | {r['face']} | {r['voice']} | {r['text']} | {r['fused']} | {r['eng']}% | {r['stress']}% |\n"

    return (result_text, breakdown,
            str(analyzed), avg_eng, avg_stress, f"{EMOJI_MAP.get(top_emo,'😐')} {top_emo}",
            log_md, log_state)

def clear_session(log_state):
    return ('😶 Waiting...\nNo analysis yet', '📷 FACE: N/A\n🎤 SPEECH: N/A\n📝 TEXT: N/A',
            '0', '—', '—', '—', 'No responses yet.', [])


with gr.Blocks(css=CSS, title='🎭 Emotion Detector') as demo:

    log_state = gr.State([])

    gr.HTML('<div id="header-bar">🎭 Emotion Detector</div>')

    with gr.Row():
        btn_analyze = gr.Button('▶ Analyze Emotion', elem_id='analyze-btn', variant='primary')
        btn_clear   = gr.Button('🗑 Clear Session', variant='secondary')

    with gr.Row():
        with gr.Column():
            gr.HTML('<div class="section-title">📷 Face Detection</div>')
            face_enabled = gr.Checkbox(label='Enable Face', value=True)
            face_img = gr.Image(label='Webcam / Upload', type='numpy', height=200,
                                sources=['webcam','upload'])

        with gr.Column():
            gr.HTML('<div class="section-title">🎤 Speech Emotion</div>')
            voice_enabled = gr.Checkbox(label='Enable Voice', value=True)
            voice_file = gr.Audio(label='Record / Upload Audio', type='filepath',
                                  sources=['microphone','upload'])

        with gr.Column():
            gr.HTML('<div class="section-title">📝 Text Analysis</div>')
            text_enabled = gr.Checkbox(label='Enable Text', value=True)
            text_input = gr.Textbox(label='Type what the student said:', lines=5,
                                     placeholder='Leave empty = text contributes 0% to the result')

    with gr.Row():
        with gr.Column(scale=1):
            gr.HTML('<div class="section-title">🔮 Result</div>')
            result_md = gr.Markdown('😶 Waiting...\nNo analysis yet')

        with gr.Column(scale=1):
            gr.HTML('<div class="section-title">📊 Modality Breakdown</div>')
            breakdown_md = gr.Markdown('📷 FACE: N/A\n\n🎤 SPEECH: N/A\n\n📝 TEXT: N/A')

    gr.HTML('<div class="section-title" style="margin-top:16px">📈 Session Analytics</div>')
    with gr.Row():
        stat_analyzed   = gr.Textbox(label='Analyzed',        value='0',  interactive=False)
        stat_avg_eng    = gr.Textbox(label='Avg Engagement',  value='—',  interactive=False)
        stat_avg_stress = gr.Textbox(label='Avg Stress',      value='—',  interactive=False)
        stat_top_emo    = gr.Textbox(label='Top Emotion',     value='—',  interactive=False)

    gr.HTML('<div class="section-title" style="margin-top:16px">📋 Response Log (Anonymous)</div>')
    log_md = gr.Markdown('No responses yet.')

    outputs = [result_md, breakdown_md,
               stat_analyzed, stat_avg_eng, stat_avg_stress, stat_top_emo,
               log_md, log_state]

    btn_analyze.click(
        fn=analyze,
        inputs=[face_img, voice_file, text_input,
                face_enabled, voice_enabled, text_enabled, log_state],
        outputs=outputs
    )
    btn_clear.click(
        fn=clear_session,
        inputs=[log_state],
        outputs=outputs
    )

print('Dashboard defined. Launching...')
demo.launch(share=True, debug=True)

## Step 12: Model Summary & Final Analytics

In [ ]:
print('='*55)
print('  MULTIMODAL EMOTION SYSTEM — MODEL SUMMARY')
print('='*55)
print(f'  Face CNN      | Best Val Acc: {max(face_history["val_acc"])*100:.2f}%')
print(f'  Voice CNN     | Best Val Acc: {max(voice_history["val_acc"])*100:.2f}%')
print(f'  DistilBERT    | Best Val Acc: {max(text_history["val_acc"])*100:.2f}%')
print('  Fusion Engine | Dynamic weighted averaging')
print('='*55)
print()
print('Models saved to:', MODEL_SAVE)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('All Models — Validation Accuracy', fontsize=13, fontweight='bold')
for ax, hist, name, color in zip(axes,
    [face_history, voice_history, text_history],
    ['Face CNN', 'Voice CNN', 'DistilBERT (Text)'],
    ['#7c3aed', '#f59e0b', '#22c55e']):
    ax.plot(hist['train_acc'], label='Train', color=color, linestyle='--', alpha=0.7)
    ax.plot(hist['val_acc'],   label='Val',   color=color)
    ax.set_title(name); ax.legend(); ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('/content/all_models_summary.png', dpi=100)
plt.show()
print('Summary chart saved to /content/all_models_summary.png')